# Drone / UAS Detection — Dataset Exploration & Inference Demo

**KTP Associate PoC | University of Central Lancashire × Operational Solutions Ltd**

This notebook:
1. Loads sample frames from `data/sample_clips/`
2. Displays 5 representative frames with matplotlib
3. Runs YOLOv8 inference on those frames
4. Visualises detection results (bounding boxes + confidence)
5. Prints basic dataset statistics: frame count, resolution, detection count per frame

Populate `data/sample_clips/` with `.jpg` or `.png` frames extracted from UAS footage before running.
The notebook handles an empty directory gracefully.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys
import glob
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from ultralytics import YOLO

# Ensure we can import from src/ if needed
sys.path.insert(0, str(Path("../src").resolve()))

print("Imports OK")

In [ ]:
# ── Device selection ─────────────────────────────────────────────────────────
# Mirrors the logic in detect_track.py so notebook inference uses the same
# hardware as the production pipeline.

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE.upper()}")

In [ ]:
# ── 1. Load sample frames ────────────────────────────────────────────────────
# Discover image files in data/sample_clips/.
# If the directory is empty or absent, we generate synthetic placeholder frames
# so the notebook remains runnable without real data.

SAMPLE_DIR = Path("../data/sample_clips")
EXTS = (".jpg", ".jpeg", ".png", ".bmp")

def load_sample_frames(sample_dir: Path, max_frames: int = 20) -> list:
    """Load up to max_frames images from sample_dir as BGR numpy arrays."""
    if not sample_dir.exists():
        return []
    files = sorted(
        p for p in sample_dir.iterdir()
        if p.suffix.lower() in EXTS
    )
    frames = []
    for fp in files[:max_frames]:
        img = cv2.imread(str(fp))
        if img is not None:
            frames.append((fp.name, img))
    return frames


def make_placeholder_frames(n: int = 5) -> list:
    """
    Generate synthetic gradient frames for demonstration when no real data
    is available. Each frame simulates a sky background with a random dark
    rectangle representing a distant drone silhouette.
    """
    frames = []
    rng = np.random.default_rng(42)
    for i in range(n):
        # Sky-blue gradient background
        img = np.zeros((480, 640, 3), dtype=np.uint8)
        for row in range(480):
            intensity = int(150 + (row / 480) * 80)
            img[row, :] = [intensity, intensity + 20, min(intensity + 60, 255)]

        # Synthetic drone silhouette (dark rectangle)
        x = int(rng.integers(50, 550))
        y = int(rng.integers(50, 380))
        w = int(rng.integers(20, 60))
        h = int(rng.integers(8, 20))
        cv2.rectangle(img, (x, y), (x + w, y + h), (20, 20, 20), -1)

        frames.append((f"placeholder_{i+1:03d}.png", img))
    return frames


raw_frames = load_sample_frames(SAMPLE_DIR)

if not raw_frames:
    print(f"[Warning] No images found in {SAMPLE_DIR}. Using synthetic placeholder frames.")
    raw_frames = make_placeholder_frames(5)
    using_placeholders = True
else:
    print(f"[OK] Loaded {len(raw_frames)} frames from {SAMPLE_DIR}")
    using_placeholders = False

In [ ]:
# ── 2. Display 5 sample frames ───────────────────────────────────────────────
# Show up to 5 frames in a grid to give a quick visual overview of the dataset.
# Frames are converted from BGR (OpenCV default) to RGB for matplotlib display.

display_frames = raw_frames[:5]
n_display = len(display_frames)

fig, axes = plt.subplots(1, n_display, figsize=(4 * n_display, 4))
if n_display == 1:
    axes = [axes]

for ax, (fname, bgr) in zip(axes, display_frames):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.set_title(fname, fontsize=8)
    ax.axis("off")

label = "(Synthetic placeholders — populate data/sample_clips/ with real UAS footage)" if using_placeholders else "Sample frames from data/sample_clips/"
fig.suptitle(label, fontsize=10, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Load YOLOv8 and run inference ─────────────────────────────────────────
# Load the pretrained YOLOv8n model (downloaded automatically on first run).
# In a C-UAS deployment, this would be replaced with a model fine-tuned on
# a labelled drone dataset covering multiple UAS classes and conditions.

print("Loading YOLOv8n...")
model = YOLO("yolov8n.pt")
print("Model loaded.\n")

CONF_THRESHOLD = 0.25   # Lower threshold for exploration — catches more candidates

# Run inference on all loaded frames
inference_results = []

for fname, bgr in raw_frames:
    result = model.predict(
        source=bgr,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )[0]
    inference_results.append((fname, bgr, result))

print(f"Inference complete on {len(inference_results)} frames.")

In [ ]:
# ── 4. Show detection results visually ───────────────────────────────────────
# Render bounding boxes and class labels on each frame and display in a grid.
# This mirrors what operators would see in a live C-UAS monitoring interface.

def draw_detections(bgr: np.ndarray, result) -> np.ndarray:
    """Draw YOLOv8 detections onto a copy of the frame and return it."""
    annotated = bgr.copy()
    if result.boxes is None or len(result.boxes) == 0:
        return annotated

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        label = f"{result.names[cls_id]} {conf:.2f}"

        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 200, 255), 2)
        cv2.putText(
            annotated, label, (x1, max(y1 - 8, 12)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1
        )
    return annotated


display_results = inference_results[:5]
n_display = len(display_results)

fig, axes = plt.subplots(1, n_display, figsize=(4 * n_display, 4))
if n_display == 1:
    axes = [axes]

for ax, (fname, bgr, result) in zip(axes, display_results):
    annotated = draw_detections(bgr, result)
    rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.imshow(rgb)
    ax.set_title(f"{fname}\n{n_det} detection(s)", fontsize=8)
    ax.axis("off")

fig.suptitle(f"YOLOv8n Detections (conf ≥ {CONF_THRESHOLD})", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Dataset statistics ─────────────────────────────────────────────────────
# Print a summary table covering the key dataset properties needed to assess
# suitability for model training and evaluation:
#   - Total frame count
#   - Frame resolution (width × height)
#   - Per-frame detection count
# In a full C-UAS dataset, additional statistics (class distribution,
# bounding box size distribution, lighting condition tags) would be reported.

import pandas as pd

rows = []
for fname, bgr, result in inference_results:
    h, w = bgr.shape[:2]
    n_det = len(result.boxes) if result.boxes is not None else 0
    rows.append({
        "filename":    fname,
        "width":       w,
        "height":      h,
        "n_detections": n_det,
    })

df = pd.DataFrame(rows)

print("=" * 55)
print(" DATASET STATISTICS")
print("=" * 55)
print(f"  Total frames          : {len(df)}")
print(f"  Resolution            : {df['width'].iloc[0]}×{df['height'].iloc[0]} px" if len(df) > 0 else "  Resolution: N/A")
print(f"  Total detections      : {df['n_detections'].sum()}")
print(f"  Mean detections/frame : {df['n_detections'].mean():.2f}")
print(f"  Max detections/frame  : {df['n_detections'].max()}")
print(f"  Frames with 0 det.    : {(df['n_detections'] == 0).sum()}")
print("=" * 55)
print()
print("Per-frame breakdown:")
print(df.to_string(index=False))

## Notes & Next Steps

- Replace synthetic placeholder frames with real UAS footage in `data/sample_clips/`.
- Fine-tune YOLOv8 on a labelled drone dataset (e.g. DUT Anti-UAV, NPS Drone Dataset) and swap the model path above.
- Run the full pipeline with `python src/detect_track.py` to produce tracked video output.
- Evaluate with `python src/evaluate.py` against ground-truth annotations.
- Extend to thermal frames by replacing the colour input with LWIR imagery and retraining.
